## INSTALLING LIBRARIES

In [ ]:
!pip install -q langchain_community
!pip install -q faiss-cpu
!pip install -q langchain-huggingface
!pip install -q langchain_openai
!pip install -q langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 14.7 MB/s eta 0:00:00


## Importing Libraries

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import os
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## Load Text Document

In [ ]:
from langchain_community.document_loaders import TextLoader

files = [
    "Denied Claim 1.txt",
    "Denied Claim 2.txt",
    "Denied Claim 3.txt"
]

documents = []

for file in files:
    loader = TextLoader(file)
    docs = loader.load()
    documents.extend(docs)

print("Number of documents loaded:", len(documents))
print("Content:", documents[2].page_content)
print("Metadata:", documents[2].metadata)

Number of documents loaded: 3
Content: Claim ID: 158927-2025-03-18
Patient Name: Robert Nguyen
Patient Age/Gender: 65-year-old Male
Provider Name / Specialty: Dr. Sarah Kim, Orthopedic Surgery
Date of Service: March 18, 2025
CPT Code: 99215
ICD Code: M17.11 (Unilateral primary osteoarthritis, right knee)
Billed Amount: $250.00
Allowed Amount: $175.00
Partial Payment: $0.00
Denial Reason: Insufficient documentation to support high-level complexity E/M service (CO50).

Procedure Notes / Clinical Documentation:

Follow-up for right knee osteoarthritis.

Patient reports mild pain, no recent injury or flare-ups.

Physical exam: stable range of motion, mild tenderness, no swelling, gait normal.

Review of imaging: X-rays unchanged from previous study; no new findings.

Vital signs: BP 130/82 mmHg, HR 75 bpm, weight 190 lbs.

Physician notes: continue current therapy, recommend low-impact exercises, monitor symptoms.

No new medications, injections, or advanced procedures performed.

Document

## Split the document into chunks

In [ ]:
doc_splitter = RecursiveCharacterTextSplitter (chunk_size =2000, chunk_overlap= 100)
split_texts = doc_splitter.split_documents(documents)
print(len(split_texts))

4


In [ ]:
split_texts[0].page_content

'Claim ID: 145872-2025-03-12\nPatient Name: John Doe\nPatient Age/Gender: 58-year-old Male\nProvider Name / Specialty: Dr. Jane Smith, Internal Medicine\nDate of Service: March 12, 2025\nCPT Code: 99214\nICD Code: E11.9 (Type 2 Diabetes Mellitus without complications)\nBilled Amount: $210.00\nAllowed Amount: $150.00\nPartial Payment: $0.00\nDenial Reason: Insufficient documentation to support billed level of service; service considered not medically necessary per payer guidelines (CO50).\n\nProcedure Notes / Clinical Documentation:\n\nRoutine follow-up visit for stable Type 2 Diabetes.\n\nPatient reports no new symptoms; blood sugar levels stable.\n\nVital signs: BP 128/78 mmHg, HR 72 bpm, weight 185 lbs, BMI 28.4.\n\nPhysical exam: normal cardiac and pulmonary findings, no edema, extremities normal.\n\nLaboratory review: A1C 6.8%, fasting glucose 110 mg/dL.\n\nCurrent medication regimen: Metformin 1000 mg BID, lifestyle counseling continued.\n\nNo new medications prescribed, no signif

In [ ]:
split_texts[1].page_content

'If the provider believes CPT 99214 is justified, an appeal may be submitted within 180 days, including:\n\nOriginal claim\n\nComplete medical record\n\nSupplemental documentation supporting medical necessity\n\nAppeals will be re-evaluated by the clinical review team for possible reconsideration.'

# Initialize the Text Embedding Model

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Build a FAISS Vector Store

In [ ]:
vectorstore = FAISS.from_documents(split_texts, embeddings)

In [ ]:
vectorstore

# Create a Retriever

In [ ]:
retriever = vectorstore.as_retriever()

# Create a Prompt_Template

In [ ]:
prompt_template = """
You are a an assistant for question answering task. Read claim id and answer based on the claim id provided.
Use the follwing pieces of Retrieved context to answer the question.
If you don't know the answer just say that you dont know.
Use one sentence and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

## Create the Prompt

In [ ]:
prompt = PromptTemplate (template = prompt_template, input_variables= ["question", "context"])

# Initialize the Large Language Model

In [ ]:
os.environ["OPENAI_API_KEY"] = "####****"
llm = ChatOpenAI(model_name = "gpt-3.5-turbo", temperature = 0.0)

# Format Retrieved Docs into Context String

In [ ]:
def format_docs(docs):
  context= ""
  for doc in docs:
    context += doc.page_content + "\n\n"
  return context

## Build the RAG Chain

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Ask a question

In [ ]:
query= "What is the Denial reason for 152304-2025-03-15?"
response = rag_chain.invoke(query)

print("Question", query)
print("Answer", response)

Question What is the Denial reason for 152304-2025-03-15?
Answer Insufficient documentation to support high-level complexity E/M service.


In [ ]:
query= "What is the Billed Amount for Claim ID: 158927-2025-03-18?"
response = rag_chain.invoke(query)

print("Question", query)
print("Answer", response)

Question What is the Billed Amount for Claim ID: 158927-2025-03-18?
Answer The Billed Amount for Claim ID 158927-2025-03-18 is $250.00.


In [ ]:
query= "Summarize the Payer's explaination for 158927-2025-03-18"
response = rag_chain.invoke(query)

print("Question", query)
print("Answer", response)

Question Summarize the Payer's explaination for 158927-2025-03-18
Answer The payer denied the claim for insufficient documentation to support a high-level complexity E/M service.


In [ ]:
query= "When can be the appeal submitted for 145872-2025-03-12?"
response = rag_chain.invoke(query)

print("Question", query)
print("Answer", response)

Question When can be the appeal submitted for 145872-2025-03-12?
Answer An appeal for claim ID 145872-2025-03-12 can be submitted within 180 days.
